In [1]:
import pandas as pd
import json
from globals import (
    BASE_DIR, 
    top_k_eval, 
    valid_popularity, 
    recommendation_dirpart, 
    full_eval_methods, 
    boosting_methods, 
    dynamic_methods,
    models_for_recbole,
    fairness_agents,
)
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.markers import MarkerStyle
from postprocess_baseline_top_k import dataset_metadata
from evaluation_metrics import (
    ndcg, 
    calculate_arp_poplift, 
    evaluation_user_group_means, 
    jensen_shannon, 
    jensen_shannon_per_user,
    per_user_distribution_stats,
    behavioral_ild_per_user, 
    geographic_ild_per_user,
    unified_fairness_nsw,
    unified_fairness_metric_scores,
)
from provider_reranker import build_item_similarity
from civic_reranker import load_coordinates
from platform_reranker import calculate_user_popularity_distributions
import pingouin as pg

dataset = "yelp" # perform for each dataset individually
valid_metrics = ["ndcg", "poplift", "ild", "geo_ild", "gini", "js", "nsw", "fairness_l2", "fairness_chebyshev"]
print(top_k_eval)

10


##  Define Functions

In [2]:
def process_top_k_json(input_file, output_file, k=10):
    """
    Process top-k recommendations from a JSON file, keeping only the item IDs for each user.
    """
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    
    try:
        with open(input_file, "r") as infile:
            data = json.load(infile)

        top_k_result = {}
        for user_id, recommendations in data.items():
            if recommendations and isinstance(recommendations[0], dict):
                item_ids = recommendations[0]["item_id"][:k]
                top_k_result[user_id] = item_ids

        with open(output_file, "w") as outfile:
            json.dump(top_k_result, outfile, indent=4)
        print(f"Processed file saved to: {output_file}")
    
    except Exception as e:
        print(f"Error processing {input_file}: {e}")

In [3]:
def create_model_directories(dataset, data, base_dir, recommendation_dirpart):
    """
    Create output directories
    """
    model_directories = {}
    methods = full_eval_methods + dynamic_methods

    def recommender_dir_combiner(dataset, modelpart, method):
        return os.path.join(base_dir, f"{dataset}_dataset", recommendation_dirpart, modelpart, method, "top_k_recommendations.json")

    for result in data:
        model_name = result["model"]
        model_directories[model_name] = {}
        
        for method in methods:
            model_directories[model_name][method] = recommender_dir_combiner(dataset, result["directory"], method)
    
    return model_directories

def open_ground_truth_user_group(dataset, valid_popularity=valid_popularity):
    """Perform data splitting and user group creation."""

    train_data = pd.read_csv(os.path.join(BASE_DIR, f"{dataset}_dataset", "processed_data_recbole", f"{dataset}_sample.train.inter"), sep="\t")
    test_data = pd.read_csv(os.path.join(BASE_DIR, f"{dataset}_dataset", "processed_data_recbole", f"{dataset}_sample.test.inter"), sep="\t")
    valid_data = pd.read_csv(os.path.join(BASE_DIR, f"{dataset}_dataset", "processed_data_recbole", f"{dataset}_sample.valid.inter"), sep="\t")

    train_data = pd.concat([train_data, valid_data])
    user_group_dir = os.path.join(BASE_DIR, f"{dataset}_dataset", f"{dataset}_user_id_popularity.json")
    with open(user_group_dir) as f:
        user_groups = json.load(f)

    user_groups["HighPop"] = user_groups.pop("high")
    user_groups["LowPop"] = user_groups.pop("low")
    user_groups["MedPop"] = user_groups.pop("medium")



    checkin_df = train_data.copy()
    
    # Calculate item popularity
    value_counts = checkin_df["item_id:token"].value_counts().reset_index()
    value_counts.columns = ["item_id:token", "count"]
    value_counts[valid_popularity] = value_counts["count"] / len(value_counts)
    checkin_df = checkin_df.merge(
        value_counts[["item_id:token", valid_popularity]],
        on="item_id:token",
        how="left",
    )
    checkin_df.sort_values(by=valid_popularity, ascending=False, inplace=True)
    item_popularity = checkin_df.drop_duplicates(subset="item_id:token", keep="first")[
        ["item_id:token", valid_popularity]
    ]

    h_group = item_popularity.head(int(len(item_popularity) * 0.2))
    h_group["item_pop_group"] = "h"
    t_group = item_popularity.tail(int(len(item_popularity) * 0.2))
    t_group["item_pop_group"] = "t"
    m_group = item_popularity[
        ~item_popularity["item_id:token"].isin(h_group["item_id:token"]) &
        ~item_popularity["item_id:token"].isin(t_group["item_id:token"])
    ]
    m_group["item_pop_group"] = "m"

    item_popularity = pd.concat([h_group, m_group, t_group])
    item_popularity.sort_values(by=valid_popularity, inplace=True, ascending=False)

    upts = checkin_df.groupby("user_id:token")[valid_popularity].mean().reset_index()
    upts.columns = ["user_id:token", "upts"]
    return train_data, test_data, user_groups, item_popularity, upts


In [4]:
def unstack_recommendations(df):
    """Unstack the recommendations for each user into separate rows."""
    unstacked_df = df.explode(["item_id:token"]).reset_index(drop=True)
    return unstacked_df

In [5]:
def top_k_to_df(recommender_dir, top_k_eval=top_k_eval):
    """ Fixed version that handles nested lists """
    with open(recommender_dir) as f:
        data = json.load(f)
    
    base_recommendations = []
    
    for user, items in data.items():
        # FIX: Check if items is nested [[...]]
        if isinstance(items, list) and len(items) > 0 and isinstance(items[0], list):
            # Flatten: items = [[...]] -> items = [...]
            items = items[0]
        
        # Now iterate through individual items
        for item in items:
            base_recommendations.append({
                "user_id:token": user,
                "item_id:token": item
            })
    
    base_df = pd.DataFrame(base_recommendations)
    base_df = unstack_recommendations(base_df)
    df = base_df.groupby("user_id:token").head(top_k_eval)
    
    return df

In [6]:
def unified_fairness_nsw_per_user(df, stakeholder_lists, top_k_eval=top_k_eval):
    """
    Per-user unified fairness (NSW) score for one aggregation mechanism's
    top-k output `df` (columns "user_id:token"/"item_id:token", as returned
    by top_k_to_df): the Nash Social Welfare of its RBO agreement with each
    fairness stakeholder's own top-k output. Every stakeholder re-ranker's
    own list is treated as that stakeholder's ground truth, so an
    aggregation mechanism only scores well here if it stays close to ALL of
    them at once, not just the easiest one or two (see
    evaluation_metrics.unified_fairness_nsw).

    `stakeholder_lists` is {agent: {user_id: [item_ids]}}, one entry per
    globals.fairness_agents (platform/civic/provider), each built from
    top_k_to_df the same way as `df` itself.
    """
    delivered_lists = df.groupby("user_id:token")["item_id:token"].apply(list).to_dict()
    nsw_scores = {}
    for user_id, delivered in delivered_lists.items():
        sh_lists = {agent: lst.get(user_id, []) for agent, lst in stakeholder_lists.items()}
        if not any(sh_lists.values()):
            continue
        _, nsw = unified_fairness_nsw(delivered, sh_lists, k=top_k_eval)
        nsw_scores[user_id] = nsw
    return nsw_scores


def unified_fairness_metric_scores_per_user(ild_scores, geo_ild_scores, jsd_scores, geo_ild_ideal_scores):
    """
    Per-user Fairness_L2 / Fairness_Chebyshev -- compromise-programming
    distances from the ideal point (1,1,1) over each user's own
    ILD/GeoILD/JSD achievement scores (see
    evaluation_metrics.unified_fairness_metric_scores). `geo_ild_ideal_scores`
    is the Civic specialist's own per-user GeoILD -- that user's individually
    best-achievable geographic spread, used as the ratio anchor for
    s_geo_ild; falls back to the mean of that dict for any user missing from
    it (e.g. a user with fewer than 2 Civic recommendations).

    Returns (l2_scores, chebyshev_scores), each a {user_id: score} dict.
    """
    fallback_ideal = float(np.mean(list(geo_ild_ideal_scores.values()))) if geo_ild_ideal_scores else 0.0
    l2_scores, chebyshev_scores = {}, {}
    for user_id, ild in ild_scores.items():
        geo_ild = geo_ild_scores.get(user_id, 0.0)
        jsd = jsd_scores.get(user_id, 1.0)
        geo_ild_ideal = geo_ild_ideal_scores.get(user_id, fallback_ideal)
        scores = unified_fairness_metric_scores(ild, geo_ild, jsd, geo_ild_ideal)
        l2_scores[user_id] = scores["fairness_l2"]
        chebyshev_scores[user_id] = scores["fairness_chebyshev"]
    return l2_scores, chebyshev_scores

In [7]:
def create_pop_distributions(data, item_popularity, user_groups):
    """Create a DataFrame with the distribution of item popularity for different user groups."""
    data = data.merge(item_popularity, on="item_id:token", how="left")
    g1 = data.loc[data["user_id:token"].isin(user_groups["HighPop"])].value_counts("item_pop_group", normalize=True).rename("g1")
    g2 = data.loc[data["user_id:token"].isin(user_groups["MedPop"])].value_counts("item_pop_group", normalize=True).rename("g2")
    g3 = data.loc[data["user_id:token"].isin(user_groups["LowPop"])].value_counts("item_pop_group", normalize=True).rename("g3")
    all = data.value_counts("item_pop_group", normalize=True).rename("all")
    distr_df = pd.DataFrame([g1, g2, g3, all]).fillna(0)
    distr_df.rename(index={"g1":"HighPop", "g2":"MedPop", "g3":"LowPop", "all":"All"}, inplace=True)
    return distr_df

In [8]:
def preprocess_distr(distr_df):
    """Preprocess the distribution DataFrame for plotting."""
    #distr_df.rename(index={"g1":"HighPop", "g2":"MedPop", "g3":"LowPop", "all":"All"}, inplace=True) # COMMENT IN FOR CALCULATING RESULTS, COMMENT OUT FOR PLOTTING
    data = distr_df.to_dict()
    result = []
    user_groups = data["h"].keys()

    for group in user_groups:
        h_value = data.get("h", {}).get(group, 0)
        m_value = data.get("m", {}).get(group, 0)
        t_value = data.get("t", {}).get(group, 0)
        
        result.append({
            "user_group": group,
            "h_ratio": h_value,
            "m_ratio": m_value,
            "t_ratio": t_value
        })

    return result


In [9]:
# Function for plotting popularity distribution (no legend inside this function)
def plot_popularity_distribution(ax, distr_df, label):
    """ Plot the popularity distribution of items for different user groups."""
    desired_order = ["LowPop", "MedPop", "HighPop", "All"]
    distr_df = distr_df.reindex(desired_order)
    colors = plt.cm.viridis([0.1, 0.5, 0.9])
    bars = distr_df.plot(kind="bar", stacked=True, ax=ax, color=colors, legend=False, edgecolor="black", linewidth=0, width=0.6)
    if label is not None:
        ax.set_title(f"{label}", fontsize=10)
    ax.set_xlabel("User Groups")
    return distr_df


In [10]:
def calculate_t_test_between_user_groups(group_scores):
    """ Calculate t-test for two-sample t-test between low and high groups. 
    Source t-test: https://www.geeksforgeeks.org/how-to-conduct-a-two-sample-t-test-in-python/"""
    
    ttest = {}
    # Conducting two-sample ttest
    result_low_high = pg.ttest(list(group_scores["LowPop"].values()), 
                    list(group_scores["HighPop"].values()),
                    correction=True)
    
    result_low_med = pg.ttest(list(group_scores["LowPop"].values()), 
                    list(group_scores["MedPop"].values()),
                    correction=True)
    
    ttest["low_high"] = float(result_low_high["p-val"])
    ttest["low_medium"] = float(result_low_med["p-val"])

    return ttest


In [11]:
def t_tests(group_scores, groups=("All", "HighPop", "MedPop", "LowPop")):
    variants = full_eval_methods # + dynamic_methods
    metrics = ["ndcg", "poplift", "ild", "geo_ild"]
    ttest_results = {}

    def clean(d):
        return [float(v) for v in d.values() if isinstance(v, (float, int))]

    for model_name, methods in group_scores.items():
        ttest_results[model_name] = {}

        for metric in metrics:
            ttest_results[model_name][metric] = {}

            for group in groups:
                ttest_results[model_name][metric][group] = {}

                baseline_vals = clean(
                    methods.get("baseline", {}).get(metric, {}).get(group, {})
                )

                for variant in variants:
                    variant_vals = clean(
                        methods.get(variant, {}).get(metric, {}).get(group, {})
                    )

                    if baseline_vals and variant_vals:
                        result = pg.ttest(baseline_vals, variant_vals, correction=True, paired=True)
                        ttest_results[model_name][metric][group][variant] = {
                            "p_val": float(result["p-val"].values[0]),
                            "delta": float(np.mean(variant_vals) - np.mean(baseline_vals)),
                        }
                    else:
                        ttest_results[model_name][metric][group][variant] = {
                            "p_val": None,
                            "delta": None,
                        }

    return ttest_results

In [12]:
def filter_to_group(general_results, ttest_rq2, metrics_to_keep, group="All"):
    """Filters data down to only the given user group and removes 'arp' before formatting."""
    filtered_data = {}


    for model in general_results:
        filtered_data[model] = {}

        for method in general_results[model]:
            if group not in general_results[model][method]:
                continue

            filtered_data[model][method] = {}

            for metric in metrics_to_keep:
                raw_val = general_results[model][method][group].get(metric)
                if raw_val is None:
                    continue

    
                t_data = None
                if method != "baseline" and metric in [
                    "ndcg",
                    "poplift",
                    "ild",
                    "geo_ild",
                ]:
                    try:
                        t_data = ttest_rq2[model][metric][group][method]
                    except KeyError:
                        pass

                filtered_data[model][method][metric] = {
                    "val": raw_val,
                    "t_test": t_data,
                }

    return filtered_data

##  Run Main Evaluation

In [13]:
data = dataset_metadata(dataset, recommendation_dirpart)
model_dirs = create_model_directories(dataset, data, BASE_DIR, recommendation_dirpart)
train_data, test_data, user_groups, item_popularity, upts = open_ground_truth_user_group(dataset)
user_groups["All"] = user_groups["HighPop"] + user_groups["MedPop"] + user_groups["LowPop"]
ground_truth_distr = create_pop_distributions(train_data, item_popularity, user_groups)
poi_df = load_coordinates(dataset)
item_sim_matrix, item_idx = build_item_similarity(train_data)
total_catalog_size = train_data["item_id:token"].nunique()
item_coords = dict(zip(poi_df["item_id:token"], zip(poi_df["lat:float"], poi_df["lon:float"])))
distr_dict_ground_truth = preprocess_distr(ground_truth_distr)

# per-user h/m/t popularity profile (own check-in history), reused for per-user JSD below
train_data_with_pop = train_data.merge(item_popularity, on="item_id:token", how="left")
user_profiles = calculate_user_popularity_distributions(train_data_with_pop, item_popularity)

group_scores = {}
results = {}
ttest_results = {}

for model_name, methods in model_dirs.items():
    if model_name in models_for_recbole:
        results[model_name] = {}
        ttest_results[model_name] = {}
        group_scores[model_name] = {}

        # Each fairness stakeholder's own top-k output, treated as that
        # stakeholder's ground truth for the unified fairness metrics (NSW,
        # L2, Chebyshev) -- computed once per model, reused for every
        # method_name below.
        stakeholder_lists = {
            agent: top_k_to_df(methods[agent]).groupby("user_id:token")["item_id:token"].apply(list).to_dict()
            for agent in fairness_agents
            if agent in methods
        }

        # Civic specialist's own per-user GeoILD, used as the s_geo_ild ratio
        # anchor (that user's individually best-achievable geographic spread)
        # for the Fairness_L2/Chebyshev metrics below -- computed once per
        # model, same reuse pattern as stakeholder_lists.
        civic_geo_ild_ideal = (
            geographic_ild_per_user(top_k_to_df(methods["civic"]), item_coords)
            if "civic" in methods else {}
        )

        for method_name, json_file in methods.items():  
            df = top_k_to_df(json_file)
            df_with_pop = df.merge(item_popularity, on="item_id:token", how="left")
            arp_scores, poplift_scores = calculate_arp_poplift(
                df_with_pop, item_popularity, upts, valid_popularity
            )

            per_user = {
                "ndcg":           ndcg(test_data=test_data, df=df, top_k_eval=top_k_eval),
                "arp":            arp_scores,
                "poplift":        poplift_scores,
                "ild":            behavioral_ild_per_user(df, item_sim_matrix, item_idx),
                "geo_ild":        geographic_ild_per_user(df, item_coords),
                "jsd_user":       jensen_shannon_per_user(user_profiles, df_with_pop),
            }

            per_user["nsw"] = unified_fairness_nsw_per_user(df, stakeholder_lists, top_k_eval)
            l2_scores, chebyshev_scores = unified_fairness_metric_scores_per_user(
                per_user["ild"], per_user["geo_ild"], per_user["jsd_user"], civic_geo_ild_ideal
            )
            per_user["fairness_l2"] = l2_scores
            per_user["fairness_chebyshev"] = chebyshev_scores
            
            distr_dict_recs = preprocess_distr(create_pop_distributions(df, item_popularity, user_groups))
            jsd_group = {
                group_gt["user_group"]: jensen_shannon(group_gt, group_recs)
                for group_gt, group_recs in zip(distr_dict_ground_truth, distr_dict_recs)
            }

            group_eval, per_user_by_group = evaluation_user_group_means(
                per_user, user_groups, df,
                total_catalog_size=total_catalog_size,
            )
            for group_name, jsd_value in jsd_group.items():
                group_eval.setdefault(group_name, {})["js"] = jsd_value


            group_scores[model_name][method_name] = per_user_by_group
            results[model_name][method_name] = group_eval

Loaded 4515 POI coordinates for dataset 'yelp'
Interaction matrix shape: (1500, 4432)
Item similarity matrix shape: (4432, 4432)


In [14]:
rename_metrics = {"ndcg": "nDCG↑", "poplift" : "PopLift→0", "ild":"ILD↑", "geo_ild" : "GeoILD↓", "gini" : "Gini↓", "js" : "JSD↓", "nsw": "NSW↑", "fairness_l2": "Fairness L2↓", "fairness_chebyshev": "Fairness L∞↓"}
rename_methods = {"baseline" : "Baseline", 
                  "platform" : "Platform", 
                  "provider" : "Provider", 
                  "civic" : "Civic", 
                  "borda" : "Borda (static)", 
                  "schulze" : "Schulze (static)",
                  "rrf" : "Reciprocal Rank Fusion",
                  "mo_greedy" : "Greedy Weighted Sum (minmax normalized)",
                  "mo_greedy_pctrank" : "Greedy Weighted Sum (percentile rank normalized)", 
                  "borda_weighted_mi" : "Borda-Weighted (m_i)",
                  "schulze_weighted_mi" : "Schulze-Weighted (m_i)",
                  "borda_weighted_ci" : "Borda-Weighted (c_i)",
                  "schulze_weighted_ci" : "Schulze-Weighted (c_i)",
                   'borda_weighted_mi_ci' : "Borda-Weighted (m_i & c_i)",
                   'schulze_weighted_mi_ci' : "Schulze-Weighted (m_i & c_i)",
                  "borda_leastfair" : "Borda-LeastFair",
                  "schulze_leastfair" : "Schulze-LeastFair",
                  "borda_lottery_mi" : "Borda-Lottery (m_i)",
                  "borda_lottery_ci" : "Borda-Lottery (c_i)",
                  "borda_lottery_mi_ci" : "Borda-Lottery (m_i & c_i)",
                  "schulze_lottery_mi" : "Schulze-Lottery (m_i)",
                  "schulze_lottery_ci" : "Schulze-Lottery (c_i)",
                  "schulze_lottery_mi_ci" : "Schulze-Lottery (m_i & c_i)",
                  }

In [15]:
ttest_rq2 = t_tests(group_scores)

user_groups_to_report = ["All", "HighPop", "MedPop", "LowPop"]
filtered_results_by_group = {}
df_group_list = []  # <-- collect per-group dataframes here

for group in user_groups_to_report:
    filtered_results = filter_to_group(results, ttest_rq2, valid_metrics, group=group)
    filtered_results_by_group[group] = filtered_results

    rows = []
    for model, methods in filtered_results.items():
        for method, metrics in methods.items():
            row_data = {"model": model, "method": method}

            for metric, data in metrics.items():
                cur_val = data["val"]
                if method == "baseline":
                    row_data[metric] = f"{cur_val:.4f}"
                    if metric == "geo_ild":
                        row_data[metric] = f"{cur_val:.2f}"
                else:
                    delta_str = ""

                    if data["t_test"] and data["t_test"].get("p_val") is not None:
                        if data["t_test"]["p_val"] < (0.05 / (26 * 4)):
                            delta_str += "*"

                    if metric == "geo_ild":
                        row_data[metric] = f"{cur_val:.2f}{delta_str}"
                    else:
                        row_data[metric] = f"{cur_val:.4f}{delta_str}"

            rows.append(row_data)

    df_group = pd.DataFrame(rows).set_index(["model", "method"])
    df_group = df_group.rename(columns=rename_metrics, index=rename_methods)
    df_group.insert(0, "group", group)   # <-- tag rows with the group name

    df_group_list.append(df_group)

    print(f"\n{'='*60}\n{group} user group\n{'='*60}")
    print(df_group.to_string())
    print(df_group.to_latex(
        multirow=False,
        caption=f"Evaluation results for all methods across models ({group} user group)",
        label=f"tab:results_{group.lower()}",
        escape=False,
    ))

# Combined dataframe, all groups in one table
df_all_groups = pd.concat(df_group_list)

# Keep "filtered_results" pointing at the "All" group, for the plotting cells below that expect it
filtered_results = filtered_results_by_group["All"]

/home/aforster/dev/multistakeholder_poi/venv/lib/python3.11/site-packages/pingouin/parametric.py:248: UserWarning: x and y are equals. Cannot compute T or p-value.
  warnings.warn("x and y are equals. Cannot compute T or p-value.")
/home/aforster/dev/multistakeholder_poi/venv/lib/python3.11/site-packages/pingouin/parametric.py:248: UserWarning: x and y are equals. Cannot compute T or p-value.
  warnings.warn("x and y are equals. Cannot compute T or p-value.")
/home/aforster/dev/multistakeholder_poi/venv/lib/python3.11/site-packages/pingouin/parametric.py:248: UserWarning: x and y are equals. Cannot compute T or p-value.
  warnings.warn("x and y are equals. Cannot compute T or p-value.")
/home/aforster/dev/multistakeholder_poi/venv/lib/python3.11/site-packages/pingouin/parametric.py:248: UserWarning: x and y are equals. Cannot compute T or p-value.
  warnings.warn("x and y are equals. Cannot compute T or p-value.")
/home/aforster/dev/multistakeholder_poi/venv/lib/python3.11/site-package


All user group
                                                       group    nDCG↑ PopLift→0     ILD↑  GeoILD↓   Gini↓    JSD↓    NSW↑ Fairness L2↓ Fairness L∞↓
model method                                                                                                                                       
BPR   Baseline                                           All   0.0419    2.0039   0.8129    21.72  0.9703  0.2545  0.3776       0.5601       0.2945
      Platform                                           All  0.0338*   0.9643*  0.8677*    23.46  0.9281  0.0132  0.4169       0.5279       0.3008
      Provider                                           All  0.0220*   0.2747*  0.9854*  111.99*  0.8530  0.0248  0.4179       0.5599       0.3200
      Civic                                              All  0.0279*   0.8209*  0.8737*    0.61*  0.9324  0.0738  0.4399       0.1313       0.0677
      Borda (static)                                     All  0.0333*   1.2306*  0.8756*    7.39

In [16]:
df_group["group"] = group
df_group = df_group.set_index("group", append=True).reorder_levels(["group", "model", "method"])

In [17]:
df_all = df_all_groups[df_all_groups["group"] == "All"]

In [18]:
df_all

group    nDCG↑  \
model method                                                            
BPR   Baseline                                           All   0.0419   
      Platform                                           All  0.0338*   
      Provider                                           All  0.0220*   
      Civic                                              All  0.0279*   
      Borda (static)                                     All  0.0333*   
      Schulze (static)                                   All  0.0346*   
      Reciprocal Rank Fusion                             All  0.0346*   
      Greedy Weighted Sum (minmax normalized)            All   0.0403   
      Greedy Weighted Sum (percentile rank normalized)   All  0.0279*   
      Borda-LeastFair                                    All   0.0334   
      Borda-Weighted (m_i)                               All   0.0333   
      Borda-Weighted (c_i)                               All   0.0328   
      Borda-Weighted (m_i & c_i)                         All   0.0325   
      Borda-Lottery (m_i)                                All   0.0348   
      Borda-Lottery (c_i)                                All   0.0341   
      Borda-Lottery (m_i & c_i)                          All   0.0330   
      Schulze-LeastFair                                  All   0.0284   
      Schulze-Weighted (m_i)                             All   0.0347   
      Schulze-Weighted (c_i)                             All   0.0293   
      Schulze-Weighted (m_i & c_i)                       All   0.0291   
      Schulze-Lottery (m_i)                              All   0.0283   
      Schulze-Lottery (c_i)                              All   0.0284   
      Schulze-Lottery (m_i & c_i)                        All   0.0264   

                                                       PopLift→0     ILD↑  \
model method                                                                
BPR   Baseline                                            2.0039   0.8129   
      Platform                                           0.9643*  0.8677*   
      Provider                                           0.2747*  0.9854*   
      Civic                                              0.8209*  0.8737*   
      Borda (static)                                     1.2306*  0.8756*   
      Schulze (static)                                   1.2875*  0.8699*   
      Reciprocal Rank Fusion                             1.3304*  0.8658*   
      Greedy Weighted Sum (minmax normalized)            1.8577*  0.8363*   
      Greedy Weighted Sum (percentile rank normalized)   0.8249*  0.9221*   
      Borda-LeastFair                                     1.0534   0.8898   
      Borda-Weighted (m_i)                                1.1711   0.8818   
      Borda-Weighted (c_i)                                1.1167   0.8837   
      Borda-Weighted (m_i & c_i)                          1.0924   0.8861   
      Borda-Lottery (m_i)                                 1.1648   0.8762   
      Borda-Lottery (c_i)                                 1.0992   0.8810   
      Borda-Lottery (m_i & c_i)                           1.1104   0.8807   
      Schulze-LeastFair                                   0.6972   0.9086   
      Schulze-Weighted (m_i)                              1.2853   0.8703   
      Schulze-Weighted (c_i)                              0.8877   0.8945   
      Schulze-Weighted (m_i & c_i)                        0.8760   0.8981   
      Schulze-Lottery (m_i)                               0.6844   0.9083   
      Schulze-Lottery (c_i)                               0.6217   0.9115   
      Schulze-Lottery (m_i & c_i)                         0.6307   0.9109   

                                                        GeoILD↓   Gini↓  \
model method                                                              
BPR   Baseline                                            21.72  0.9703   
      Platform                                            23.46  0.9281   
   



Foursquare: 

Weighted methods are very similar to static aggregation (Borda lower nDCG and overall better beyond-accuracy metrics)
This flips in Lottery and LeastFair: Borda has higher nDCG and worse beyond-accuracy values (except for GeoILD where both Lottery allocations have close values)


Differences between c_i, m_i, and both:
Borda weighted: "c_i" + "m_i & c_i" very similar --> m_i retains higher accuracy, while including c_i favors more fairness intervention (at reduced nDCG)
Schulze weighted: "m_i" + "c_i" perform similarly, "m_i & c_i" decrease accuracy and slightly improve beyond-accuracy metrics

Borda lottery: generally small differences; "c_i" has highest nDCG and lowest PopLift
Schulze lottery: values very close, "m_i" has highest nDCG and worst beyond-accuracy metrics except for GeoILD


Yelp:

Again, weighted similar to static and in lottery the Borda and Schulze flip: Schulze lower nDCG and higher beyond-accuracy and Borda vice-versa. 
In Borda and Schulze, Lottery and LeastFair have better equality (Gini) than Weighted and static

Borda weighted: Close to Borda (static) except for GeoILD (higher avg distances); Borda Lottery has much higher GeoILD than Borda-Weighted, Borda (static) has best GeoILD values, followed by Borda (weighted) variants

Schulze weighted: Close to Schulze (static)

Borda Lottery: Borda Lottery "m_i" highest none-baseline nDCG, some beyond-accuracy metrics better than static but GeoILD very high
Schulze Lottery: Similar to Schulze LeastFair, among best beyond-accuracy metrics except for GeoILD

Differences between c_i, m_i, and both:
Borda-weighted: "m_i" slightly better nDCG and best in some beyond-accuracy metrics (GeoILD, PopLift, ILD)
Schulze-weighted: "c_i" best nDCG

Schulze-Lottery: "c_i" alone has worst nDCG



In [19]:
df_all_groups[df_all_groups["group"] == "LowPop"]

group   nDCG↑  \
model method                                                             
BPR   Baseline                                          LowPop  0.0221   
      Platform                                          LowPop  0.0278   
      Provider                                          LowPop  0.0231   
      Civic                                             LowPop  0.0226   
      Borda (static)                                    LowPop  0.0197   
      Schulze (static)                                  LowPop  0.0178   
      Reciprocal Rank Fusion                            LowPop  0.0216   
      Greedy Weighted Sum (minmax normalized)           LowPop  0.0199   
      Greedy Weighted Sum (percentile rank normalized)  LowPop  0.0208   
      Borda-LeastFair                                   LowPop  0.0248   
      Borda-Weighted (m_i)                              LowPop  0.0202   
      Borda-Weighted (c_i)                              LowPop  0.0218   
      Borda-Weighted (m_i & c_i)                        LowPop  0.0215   
      Borda-Lottery (m_i)                               LowPop  0.0289   
      Borda-Lottery (c_i)                               LowPop  0.0201   
      Borda-Lottery (m_i & c_i)                         LowPop  0.0256   
      Schulze-LeastFair                                 LowPop  0.0231   
      Schulze-Weighted (m_i)                            LowPop  0.0180   
      Schulze-Weighted (c_i)                            LowPop  0.0227   
      Schulze-Weighted (m_i & c_i)                      LowPop  0.0211   
      Schulze-Lottery (m_i)                             LowPop  0.0311   
      Schulze-Lottery (c_i)                             LowPop  0.0241   
      Schulze-Lottery (m_i & c_i)                       LowPop  0.0292   

                                                       PopLift→0     ILD↑  \
model method                                                                
BPR   Baseline                                            1.7702   0.8124   
      Platform                                           0.6161*  0.8513*   
      Provider                                           0.4599*  0.9835*   
      Civic                                              0.7692*  0.8694*   
      Borda (static)                                     1.0428*  0.8776*   
      Schulze (static)                                   1.0359*  0.8751*   
      Reciprocal Rank Fusion                             1.0804*  0.8750*   
      Greedy Weighted Sum (minmax normalized)            1.5774*  0.8464*   
      Greedy Weighted Sum (percentile rank normalized)   0.6816*  0.9228*   
      Borda-LeastFair                                     0.9053   0.8845   
      Borda-Weighted (m_i)                                1.0216   0.8812   
      Borda-Weighted (c_i)                                1.1326   0.8470   
      Borda-Weighted (m_i & c_i)                          1.1185   0.8483   
      Borda-Lottery (m_i)                                 1.0013   0.8721   
      Borda-Lottery (c_i)                                 1.0598   0.8474   
      Borda-Lottery (m_i & c_i)                           1.0301   0.8474   
      Schulze-LeastFair                                   0.6255   0.9093   
      Schulze-Weighted (m_i)                              1.0322   0.8757   
      Schulze-Weighted (c_i)                              0.6730   0.8517   
      Schulze-Weighted (m_i & c_i)                        0.6963   0.8569   
      Schulze-Lottery (m_i)                               0.6208   0.8975   
      Schulze-Lottery (c_i)                               0.6758   0.8664   
      Schulze-Lottery (m_i & c_i)                         0.6067   0.8685   

                                                        GeoILD↓   Gini↓  \
model method                                                              
BPR   Baseline                                            11.28  0.9649   
      Platform                                         

## Plots

In [20]:
local_rename_metrics = {**rename_metrics, "jsd_user": "JSD (per-user)\u2193"}
plot_metrics = ["ild", "geo_ild", "jsd_user"]
plot_methods = [m for m in model_dirs[model_name] if m in group_scores[model_name]]
plot_methods = full_eval_methods # + dynamic_methods


fig, axs = plt.subplots(1, len(plot_metrics), figsize=(5.5 * len(plot_metrics), 5.5))

for ax, metric in zip(axs, plot_metrics):
    data, labels = [], []
    for method_name in plot_methods:
        scores = group_scores[model_name][method_name].get(metric, {}).get("All", {})
        if not scores:
            continue
        data.append(list(scores.values()))
        labels.append(rename_methods.get(method_name, method_name))

    positions = range(1, len(labels) + 1)
    ax.violinplot(data, positions=positions, showmedians=True, showextrema=False)
    ax.boxplot(data, positions=positions, widths=0.15, showfliers=True)
    ax.set_xticks(list(positions))
    ax.set_xticklabels(labels, rotation=60, ha="right")
    ax.set_title(local_rename_metrics.get(metric, metric))

fig.suptitle(f"Per-user distribution by condition -- {model_name} ({dataset})")
fig.tight_layout()

plt.savefig(os.path.join(BASE_DIR, f"{dataset}_dataset", "plots", f"{dataset}_statistics.png"), bbox_inches="tight")
plt.show()


KeyError: 'NeuMF'

In [ ]:
model_to_plot = models_for_recbole[0]
variant_styles = {
    # "platform": {"color": "#1f77b4", "ls": "-", "marker": "o", "fill": "full", "label": "Platform"},
    # "provider": {"color": "#e377c2", "ls": "-", "marker": "s", "fill": "fill", "label": "Provider"},
    # "civic": {"color": "#2ca02c", "ls": "-", "marker": "D", "fill": "full", "label": "Civic"},
    "borda": {"color": "#d01414", "ls": ":", "marker": "h", "fill": "none", "label": "Borda"},
    "schulze": {"color": "#df8302", "ls": "--", "marker": "d", "fill": "none", "label": "Schulze"},
    "borda_weighted_mi_ci": {"color": "#8c564b", "ls": "-.", "marker": "^", "fill": "none", "label": "Borda-Weighted (m_i & c_i)"},
    "borda_lottery_mi_ci": {"color": "#8c564b", "ls": ":", "marker": "v", "fill": "full", "label": "Borda-Lottery (m_i & c_i)"},
    "borda_leastfair": {"color": "#8c564b", "ls": "-", "marker": "o", "fill": "full", "label": "Borda-LeastFair"},
    "schulze_weighted_mi_ci": {"color": "#17becf", "ls": "-.", "marker": "^", "fill": "none", "label": "Schulze-Weighted (m_i & c_i)"},
    "schulze_lottery_mi_ci": {"color": "#17becf", "ls": ":", "marker": "v", "fill": "full", "label": "Schulze-Lottery (m_i & c_i)"},
    "schulze_leastfair": {"color": "#17becf", "ls": "-", "marker": "o", "fill": "full", "label": "Schulze-LeastFair"},

}

rows = []
baseline_data = filtered_results[model_to_plot]["baseline"]

# Enforce loop consistency by utilizing the explicit list ordering
methods_keys = [m for m in variant_styles.keys() if m in filtered_results[model_to_plot]]

for method in methods_keys:
    delta = {}
    for m in valid_metrics:
        base_val = baseline_data[m]["val"]
        curr_val = filtered_results[model_to_plot][method][m]["val"]
        delta[m] = ((curr_val - base_val) / abs(base_val)) * 100 if base_val != 0 else 0

    delta["method"] = method
    rows.append(delta)

rel_df = pd.DataFrame(rows).set_index("method")
metric_labels = [rename_metrics.get(m, m) for m in valid_metrics]
fig, ax = plt.subplots(figsize=(11, 10.5))
CLIP_low = 100
CLIP_high = 50
Y_SPACING = 0.6
y_pos = np.arange(len(valid_metrics)) * Y_SPACING
n_methods = len(methods_keys)

for i, method in enumerate(methods_keys):
    vals = rel_df.loc[method, valid_metrics].values
    style = variant_styles[method]
    
    color = style["color"]
    ls = style["ls"]
    m_type = style["marker"]

    offset = (i - n_methods / 2 + 0.5) * 0.055

    for j, val in enumerate(vals):
        y = y_pos[j] + offset
        clipped = np.clip(val, -CLIP_low, CLIP_high)
        final_marker = "^" if val > CLIP_high else ("v" if val < -CLIP_low else m_type)
        ax.plot([0, clipped], [y, y], color=color, linestyle=ls, lw=2.5, alpha=0.8)

        if style["fill"] == "none" and final_marker == m_type:
            ax.scatter(clipped, y, edgecolor=color, facecolor="white", s=70, zorder=3, marker=final_marker, lw=1.5)
        elif style["fill"] == "left" and final_marker == m_type:
            ax.scatter(clipped, y, s=70, zorder=3, marker=MarkerStyle(final_marker, fillstyle="left"), color=color)
        else:
            ax.scatter(clipped, y, color=color, s=70, zorder=3, marker=final_marker, alpha=1.0)

ax.axvline(0, color="black", lw=1.2, ls="-", alpha=0.8)

ax.set_yticks(y_pos)
ax.set_yticklabels(metric_labels, fontsize=16)
ax.invert_yaxis()

ax.set_xlabel(f"% change compared to the {model_to_plot} baseline; large deviations clipped ▼▲", fontsize=15, labelpad=12)
ax.set_xlim(-CLIP_low-2, CLIP_high+2)
ax.tick_params(axis="x", labelsize=16, length=6, width=1.2)
ax.grid(axis="x", linestyle=":", alpha=0.5)
ax.spines[["top", "right"]].set_visible(False)

legend_handles = []
for m in methods_keys:
    style = variant_styles[m]
    label_text = style["label"]

    if style["fill"] == "none":
        handle = mlines.Line2D([], [], color=style["color"], linestyle=style["ls"], 
                               marker=style["marker"], markerfacecolor="white", markeredgecolor=style["color"],
                               markersize=9, lw=1.5, label=label_text)
    elif style["fill"] == "left":
        handle = mlines.Line2D([], [], color=style["color"], linestyle=style["ls"], 
                               marker=style["marker"], markerfacecolor=style["color"], markeredgecolor=style["color"],
                               fillstyle="left", markersize=9, lw=1.5, label=label_text)
    else:
        handle = mlines.Line2D([], [], color=style["color"], linestyle=style["ls"], 
                               marker=style["marker"], markerfacecolor=style["color"],
                               markersize=9, lw=1.5, label=label_text)
    legend_handles.append(handle)

ax.legend(
    handles=legend_handles,
    loc="upper center",
    ncol=3,
    bbox_to_anchor=(0.5, -0.13),
    frameon=True,
    fontsize=16,
)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, f"{dataset}_dataset", "plots", f"{dataset}_metrics_percentage_change.png"), bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
# Additional Config for a Relative Plot including the weighted experiments

# variant_styles_full_plot = {
#     "platform": {"color": "#1f77b4", "ls": "-", "marker": "o", "fill": "full", "label": "Platform"},
#     "provider": {"color": "#e377c2", "ls": "-", "marker": "s", "fill": "full", "label": "Provider"},
#     "civic": {"color": "#2ca02c", "ls": "-", "marker": "D", "fill": "full", "label": "Civic"},
    
#     "borda": {"color": "#d87192", "ls": "-", "marker": "h", "fill": "full", "label": "Borda"},
#     "schulze": {"color": "#df8302", "ls": "-", "marker": "d", "fill": "full", "label": "Schulze"},

#     "borda2baseline": {"color": "#7f7f7f", "ls": "--", "marker": "h", "fill": "none", "label": "Borda-User"},
#     "borda2platform": {"color": "#1f77b4", "ls": "--", "marker": "h", "fill": "none", "label": "Borda-Platform"},
#     "borda2provider": {"color": "#e377c2", "ls": "--", "marker": "h", "fill": "none", "label": "Borda-Provider"},
#     "borda2civic": {"color": "#2ca02c", "ls": "--", "marker": "h", "fill": "none", "label": "Borda-Civic"},

#     "schulze2baseline": {"color": "#7f7f7f", "ls": ":", "marker": "d", "fill": "full", "label": "Schulze-User"},
#     "schulze2platform": {"color": "#1f77b4", "ls": ":", "marker": "d", "fill": "full", "label": "Schulze-Platform"},
#     "schulze2provider": {"color": "#e377c2", "ls": ":", "marker": "d", "fill": "full", "label": "Schulze-Provider"},
#     "schulze2civic": {"color": "#2ca02c", "ls": ":", "marker": "d", "fill": "full", "label": "Schulze-Civic"},

# }

In [ ]:
method_names_for_title = ["$Base$", "$CP_\Im$", "$MMR$", "$Geo$", "$Borda$", "$Schulze$",
                           "$Borda_W$", "$Borda_L$", "$Schulze_W$", "$Schulze_L$"]
extra_dynamic_methods = ["borda_weighted_mi_ci", "borda_lottery_mi_ci", "schulze_weighted_mi_ci", "schulze_lottery_mi_ci"]
plot_methods_popdist = full_eval_methods # + extra_dynamic_methods

filtered_models = [
    (model_name, methods)
    for model_name, methods in model_dirs.items()
    if model_name in models_for_recbole
]

n_rows = len(filtered_models)
n_cols = len(plot_methods_popdist) + 1
fig, axs = plt.subplots(n_rows, n_cols, figsize=(15 * n_cols / 7, 5.5 * n_rows), squeeze=False)

for row, (model_name, methods) in enumerate(filtered_models):
    ax = axs[row, 0]
    plot_popularity_distribution(ax, ground_truth_distr, None)
    if row == 0:
        ax.set_title("User Profile", fontsize=11)
    ax.set_ylabel(model_name, fontsize=10, weight="bold", labelpad=0)

    col = 0
    for method_name in plot_methods_popdist:
        json_file = methods[method_name]
        df = top_k_to_df(json_file, top_k_eval=top_k_eval)
        distr_df = create_pop_distributions(df, item_popularity, user_groups)

        ax = axs[row, col + 1]
        plot_popularity_distribution(ax, distr_df, None)
        if row == 0:
            ax.set_title(method_names_for_title[col], fontsize=11)
        col += 1

for ax in axs.flat:
    ax.set_xticks(range(4))
    ax.set_xticklabels(["LowPop", "MedPop", "HighPop", "All"], rotation=45, fontsize=11)

for row in range(n_rows):
    for ax in axs[row, 1:]:
        ax.set_yticks([])

fig.text(0.55, 0.03, f"User Groups ({dataset.capitalize()})", ha="center", fontsize=11)
fig.text(0.04, 0.5, "Item Group Ratios", va="center", rotation="vertical", fontsize=11)

handles = [plt.Line2D([0], [0], color=color, lw=4) for color in plt.cm.viridis([0.9, 0.5, 0.1])]
labels = ["T", "M", "H"]
fig.legend(handles, labels, title="Item Groups", loc="center right", bbox_to_anchor=(1.07, 0.5), ncol=1, title_fontsize=11)

plt.tight_layout(rect=[0.05, 0.03, 1, 0.91])
os.makedirs(os.path.join(BASE_DIR, f"{dataset}_dataset", "plots"), exist_ok=True)
plt.savefig(os.path.join(BASE_DIR, f"{dataset}_dataset", "plots", f"{dataset}_popularity_distribution_paper.png"), bbox_inches="tight")